# Fine-tune Voxtral for Speech Recognition

In this notebook, we fine-tune [Voxtral Mini 3B](https://huggingface.co/mistralai/Voxtral-Mini-3B-2507) for automatic speech recognition using QLoRA (4-bit quantization + LoRA).

Voxtral is Mistral AI's speech model. It uses a prompt-masking strategy where the model receives audio tokens followed by a `<transcribe>` instruction, and loss is computed only on the transcription tokens.

**What we'll cover:**
- Loading the VoxPopuli dataset
- Prompt-masked data collation for Voxtral
- QLoRA fine-tuning (4-bit quantized model + LoRA adapters)
- Running inference

In [ ]:
!pip install -q transformers datasets accelerate peft mistral-common bitsandbytes

## Load Dataset

We use VoxPopuli from the ESB benchmark, resampled to 16kHz.

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("hf-audio/esb-datasets-test-only-sorted", "voxpopuli", split="test")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

train_dataset = dataset.select(range(100))
eval_dataset = dataset.select(range(100, 150))

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

## Load Model and Processor

In [ ]:
import torch
from transformers import VoxtralForConditionalGeneration, VoxtralProcessor, BitsAndBytesConfig

model_checkpoint = "mistralai/Voxtral-Mini-3B-2507"

processor = VoxtralProcessor.from_pretrained(model_checkpoint)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = VoxtralForConditionalGeneration.from_pretrained(
    model_checkpoint,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map="auto",
)
print(f"Model loaded: {model_checkpoint}")

## Data Collator

The key technique here is **prompt masking**: we construct sequences as `[AUDIO_TOKENS] <transcribe> reference_text`, then mask the prompt portion with `-100` so the loss focuses only on the transcription tokens. This is similar to instruction tuning for LLMs.

In [ ]:
class VoxtralDataCollator:
    def __init__(self, processor, model_id):
        self.processor = processor
        self.model_id = model_id

    def __call__(self, features):
        texts = [f["text"] for f in features]
        audios = [f["audio"]["array"] for f in features]

        # Build prompt: [AUDIO] tokens + <transcribe>
        prompt = self.processor.apply_transcription_request(
            language="en",
            model_id=self.model_id,
            audio=audios,
            format=["WAV"] * len(audios),
            return_tensors="pt",
        )
        passthrough = {k: v for k, v in prompt.items() if k not in ("input_ids", "attention_mask")}
        prompt_ids = prompt["input_ids"]
        prompt_attn = prompt["attention_mask"]
        B = prompt_ids.size(0)
        tok = self.processor.tokenizer

        # Tokenize transcriptions
        text_tok = tok(texts, add_special_tokens=False, padding=False, truncation=True, max_length=256, return_tensors=None)
        text_ids_list = text_tok["input_ids"]

        # Concatenate and mask prompt tokens in labels
        input_ids, attention_mask, labels = [], [], []
        for i in range(B):
            p_ids = prompt_ids[i].tolist()
            p_att = prompt_attn[i].tolist()
            t_ids = text_ids_list[i]

            ids = p_ids + t_ids + [tok.eos_token_id]
            attn = p_att + [1] * (len(t_ids) + 1)
            lab = [-100] * len(p_ids) + t_ids + [tok.eos_token_id]

            input_ids.append(ids)
            attention_mask.append(attn)
            labels.append(lab)

        # Pad to max length
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        max_len = max(len(x) for x in input_ids)

        def pad_to(seq, fill, L):
            return seq + [fill] * (L - len(seq))

        batch = {
            "input_ids": torch.tensor([pad_to(x, pad_id, max_len) for x in input_ids], dtype=torch.long),
            "attention_mask": torch.tensor([pad_to(x, 0, max_len) for x in attention_mask], dtype=torch.long),
            "labels": torch.tensor([pad_to(x, -100, max_len) for x in labels], dtype=torch.long),
        }
        for k, v in passthrough.items():
            batch[k] = v
        return batch

data_collator = VoxtralDataCollator(processor, model_checkpoint)

## QLoRA Fine-tuning

Since the model is loaded in 4-bit, we apply LoRA adapters to the attention layers while freezing the audio encoder. This is the QLoRA approach — combining quantization with LoRA for memory-efficient fine-tuning.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# Freeze audio encoder
for param in model.audio_tower.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./voxtral-finetuned-qlora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_steps=50,
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
from peft import LoraConfig, get_peft_model

model_lora = VoxtralForConditionalGeneration.from_pretrained(
    model_checkpoint,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Freeze audio encoder
for param in model_lora.audio_tower.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="SEQ_2_SEQ_LM",
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

## Inference